# NHS GP Cloud Based Telephony · **Gold layer**

**What this notebook does, in one breath:** before saying anything new, it rebuilds NHS
England's own published figures for August 2026 from my Silver tables and shows the
match, line by line. Only once that gate passes does it answer the three questions in
the README: who sits behind the missing 13.7% of practices, what happens to the 31.9% of
calls that are neither answered nor missed, and what the Monday 08:00–10:00 figure
really measures.

**Rule for this layer:** no figure leaves Gold unless the gate above it passed, or the
gap is located and written down.

## Step 0 — Tools, folders and the Silver tables

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

BRONZE_DIR = PROJECT_DIR / "data" / "bronze"
SILVER_DIR = PROJECT_DIR / "data" / "silver"
GOLD_DIR = PROJECT_DIR / "data" / "gold"
EVIDENCE_DIR = PROJECT_DIR / "evidence"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

MONTH = "2026-08"
CORE_BANDS = ["08:00-09:59", "10:00-11:59", "12:00-13:59", "14:00-15:59", "16:00-17:59", "18:00-18:29"]
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

day_time = pd.read_parquet(SILVER_DIR / "calls_day_time" / f"{MONTH}.parquet")
durations = pd.read_parquet(SILVER_DIR / "calls_durations" / f"{MONTH}.parquet")
answered = pd.read_parquet(SILVER_DIR / "calls_calls_answered" / f"{MONTH}.parquet")
participation = pd.read_parquet(SILVER_DIR / "practice_participation.parquet")
practice_dim = pd.read_parquet(SILVER_DIR / f"practice_dim_{MONTH}.parquet")
registered = pd.read_parquet(SILVER_DIR / "registered_patients.parquet")
table1 = pd.read_parquet(SILVER_DIR / "published_table1.parquet")
table2 = pd.read_parquet(SILVER_DIR / "published_table2_national_day_time.parquet")

published = (table1[(table1.publication_month == MONTH) & (table1.reference_month == MONTH)]
             .set_index(["group", "measure"])["value"])
print(f"Loaded Silver for {MONTH}: {len(day_time):,} day-time rows, {len(published)} published Table 1 values")

Loaded Silver for 2026-08: 3,798,314 day-time rows, 54 published Table 1 values


## Step 1 — Which day is the bank holiday?

August 2026 has 21 weekdays. Table 1 says it had **20 working weekdays**, so NHS England
treats one weekday as non-working: Monday 31 August, the Summer Bank Holiday in England.
I check that from the publication itself rather than hard-coding it. If the published
working-day count doesn't equal weekdays minus the bank holidays I list, the gate stops.

In [2]:
BANK_HOLIDAYS = [pd.Timestamp("2026-08-31")]  # I treat 31 Aug (Summer Bank Holiday, England) as non-working; the gate checks this against Table 1

dates = pd.date_range(f"{MONTH}-01", periods=pd.Period(MONTH).days_in_month)
weekdays_in_month = int((dates.dayofweek < 5).sum())
working_weekdays = weekdays_in_month - sum(d in dates for d in BANK_HOLIDAYS)
published_working = int(published[("Working Days", "Number of working weekdays")])
assert working_weekdays == published_working, (working_weekdays, published_working)
print(f"{weekdays_in_month} weekdays - {len(BANK_HOLIDAYS)} bank holiday = {working_weekdays} working weekdays "
      f"(published: {published_working})")

day_time["is_bank_holiday"] = day_time.date.isin(BANK_HOLIDAYS)

21 weekdays - 1 bank holiday = 20 working weekdays (published: 20)


## Step 2 — The reconciliation gate

Every published August 2026 headline I can rebuild goes in one table: published value,
my rebuild, the difference, and a status.

- **Counts** must match exactly.
- **Rates and percentages** must match at the precision NHS England published them
  (three decimal places for shares, one for the rate per 1,000).
- **EXPLAINED** means a gap I've located and can describe, but can't close from public
  files.
- **PUBLISHED LABEL ERROR** means my figure matches a *different* published row than the
  one its label says. That's the duration-percentage problem Silver found.

Anything else is a **FAIL**, and the notebook stops.

**Which answered count?** CBT003 (answered, by wait time) and CBT007 (answered, by
duration) should be equal and almost always are. In the three months where one practice
differs (Nov 2025, Dec 2025, Mar 2026), Table 1's "Calls answered" equals **CBT007**. So
the gate uses CBT007 for answered calls, and CBT003 only for the wait-time rows. In August
2026 the two are identical.

Some of these headlines aren't in Table 1: the core-hours and 8–10am inbound counts are
in Table 4b, and the Monday figure is on the publication page. I read those from the
Bronze files and say so in the `source` column.

In [3]:
def table4b_national() -> dict:
    path = next((BRONZE_DIR / "telephony" / MONTH).glob("*Publication Summary*.xlsx"))
    raw = pd.read_excel(path, sheet_name="Table 4b", header=None)
    row = raw[raw.iloc[:, 1].astype(str).eq("National")].iloc[0]
    v = pd.to_numeric(row, errors="coerce").dropna().tolist()
    return {"inbound_any": v[0], "answered_any": v[1], "inbound_core": v[3],
            "answered_core": v[4], "inbound_8_10": v[6], "answered_8_10": v[7]}


t4b = table4b_national()
inbound = day_time[day_time.indicator.eq("CBT001")].copy()
nat = durations.groupby("indicator", observed=True)["calls"].sum()
p_all = participation[participation.month.eq(MONTH)]
pd_reg = practice_dim.set_index("practice_code")["registered_patients"]

weekday_core = inbound[~inbound.weekday.isin(["Saturday", "Sunday"]) & ~inbound.is_bank_holiday]
mon_8_10 = inbound[inbound.weekday.eq("Monday") & inbound.time_band.eq("08:00-09:59")]["calls"].sum()

answered["under_2min"] = answered.upper_s.fillna(10**9) <= 120
core = answered[answered.core_category.ne("Non-Core")]
early = answered[answered.core_category.eq("Core_8_10")]

WAIT_BANDS = {"within 1 minute": (0, 60), "within 1 to 2 minutes": (61, 120),
              "within 2 to 5 minutes": (121, 300), "after waiting over 5 minutes": (301, None)}


def wait_counts(frame):
    return {k: int(frame.loc[(frame.lower_s >= lo) & ((frame.upper_s <= hi) if hi else frame.upper_s.isna()), "calls"].sum())
            for k, (lo, hi) in WAIT_BANDS.items()}


rows = []


def check(source, label, published_value, rebuilt, kind="count", note=""):
    if kind == "count":
        ok = round(published_value) == round(rebuilt)
    elif kind == "share":
        ok = round(rebuilt, 3) == round(published_value, 3)
    else:
        ok = round(rebuilt, 1) == round(published_value, 1)
    rows.append({"source": source, "measure": label, "published": published_value, "rebuilt": rebuilt,
                 "difference": rebuilt - published_value, "status": "PASS" if ok else "FAIL", "note": note})


T1 = "Table 1"
# I check coverage first: practices, patients and the two shares
check(T1, "Open active practices", published[("Coverage", "Open active practices")], len(p_all))
check(T1, "Count of practices included", published[("Coverage", "Count of practices included")], int(p_all.included.sum()))
check(T1, "Practice coverage", published[("Coverage", "Practice coverage [1]")], p_all.included.mean(), "share")
reg_all, reg_inc = pd_reg.sum(), pd_reg[practice_dim.set_index("practice_code").included].sum()
check(T1, "Registered patients at open active practices",
      published[("Coverage", "Registered patients at open active practices")], reg_all)
check(T1, "Registered patients at included practices",
      published[("Coverage", "Registered patients at included practices")], reg_inc)
check(T1, "Patient coverage", published[("Coverage", "Patient coverage")], reg_inc / reg_all, "share")
check(T1, "Number of working weekdays", published_working, working_weekdays)
# Then inbound calls and the four outcomes
total = int(inbound.calls.sum())
check(T1, "Total inbound calls", published[("Inbound calls", "Total Inbound Calls")], total)
check(T1, "Rate of inbound calls per 1,000 registered patients at included practices",
      published[("Inbound calls", "Rate of inbound calls per 1,000 patients registered at practices included")],
      total / reg_inc * 1000, "rate")
for code_, label, measure in [
    ("CBT007", "Calls answered", "Calls answered [2]"),  # I use CBT007 here because Table 1 does (see the note above the gate)
    ("CBT002", "Calls ended during the IVR stage", next(m for g, m in published.index if m.startswith("Calls ended during the IVR"))),
    ("CBT005", "Calls resulting in a call back request", "Calls resulting in a call back request [2]"),
    ("CBT006", "Call backs made", "Call backs made"),
    ("CBT004", "Missed calls", "Missed calls [2,4]"),
]:
    group = next(g for g, m in published.index if m == measure)
    check(T1, label, published[(group, measure)], int(nat[code_]))
for code_, measure in [("CBT007", "Percentage of Inbound calls answered"),
                       ("CBT002", "Percentage of Inbound calls ended during the IVR stage"),
                       ("CBT005", "Percentage of Inbound calls resulting in a call back request"),
                       ("CBT004", "Percentage of inbound calls missed")]:
    key = next((g, m) for g, m in published.index if m.startswith(measure))
    check(T1, key[1], published[key], nat[code_] / total, "share")
check(T1, "Percentage of call backs made of those requested",
      published[("Calls dealt with", "Percentage of call backs made of those requested")], nat["CBT006"] / nat["CBT005"], "share")
# Then wait times: any time, core hours and 8am-10am
for group, frame in [("Call wait times (any time)", answered),
                     ("Call wait times (core hours)", core),
                     ("Call wait times (8am-10am)", early)]:
    counts = wait_counts(frame)
    for band, n in counts.items():
        check(T1, f"{group}: answered {band}", published[(group, f"Calls answered {band}")], n)
    key = next((g, m) for g, m in published.index if g == group and m.startswith("Percentage of calls answered after waiting under 2"))
    check(T1, f"{group}: % answered after waiting under 2 minutes", published[key],
          frame.loc[frame.under_2min, "calls"].sum() / frame.calls.sum(), "share")
# Then the figures outside Table 1: Table 4b and the publication page
check("Table 4b", "Inbound calls, core hours (Mon-Fri 08:00-18:30)", t4b["inbound_core"],
      int(weekday_core[weekday_core.time_band.isin(CORE_BANDS)].calls.sum()),
      note="rebuilt excluding the bank holiday")
check("Table 4b", "Inbound calls, 08:00-10:00 Mon-Fri", t4b["inbound_8_10"],
      int(weekday_core[weekday_core.time_band.eq("08:00-09:59")].calls.sum()),
      note="rebuilt excluding the bank holiday")
check("Table 4b", "Answered calls, core hours", t4b["answered_core"], int(core.calls.sum()))
check("Table 4b", "Answered calls, 08:00-10:00", t4b["answered_8_10"], int(early.calls.sum()))
check("Publication page", "Inbound calls Monday 08:00-10:00", 2_037_943, int(mon_8_10),
      note="rebuilt INCLUDING the bank holiday (31 Aug: 70,616)")
check("Publication page", "Monday 08:00-10:00 as share of inbound", 0.075, mon_8_10 / total, "share")
# Last, I compare every date x time-band cell of Table 2
t2 = table2[table2.publication_month.eq(MONTH)]
rebuilt_grid = (inbound.groupby(["date", "time_band"], observed=True)["calls"].sum().reset_index())
rebuilt_grid = pd.concat([rebuilt_grid, inbound.groupby("date")["calls"].sum().reset_index().assign(time_band="TOTAL")])
grid = t2.merge(rebuilt_grid, on=["date", "time_band"], how="outer", suffixes=("_published", "_rebuilt"))
cells_matching = int((grid.calls_published == grid.calls_rebuilt).sum())
check("Table 2", f"Date x time-band cells matching ({len(grid)} cells)", len(grid), cells_matching)

### Duration percentages: matching against the right row

The four duration counts go through the normal check. For the percentages I test two
things: does my share match the row with **its own** label, and if not, does it match a
**different** published row? A match on a different row is a labelling error in the
publication, not a gap in my rebuild.

In [4]:
d7 = durations[durations.indicator.eq("CBT007")]
DUR_BANDS = {"1 minute or less": (0, 60), "1 to 2 minutes": (61, 120), "2 to 5 minutes": (121, 300), "over 5 minutes": (301, None)}
dur_counts = {k: int(d7.loc[(d7.lower_s >= lo) & ((d7.upper_s <= hi) if hi else d7.upper_s.isna()), "calls"].sum())
              for k, (lo, hi) in DUR_BANDS.items()}
dur_total = sum(dur_counts.values())
pub_pcts = {m.replace("Percentage of calls with a duration of ", ""): v
            for (g, m), v in published.items() if g == "Call durations" and m.startswith("Percentage")}
for band, n in dur_counts.items():
    check(T1, f"Call duration {band}", published[("Call durations", f"Duration of {band}")], n)
for band, n in dur_counts.items():
    share = n / dur_total
    own = pub_pcts[band]
    if round(share, 3) == round(own, 3):
        check(T1, f"% of calls with duration {band}", own, share, "share")
    else:
        printed_as = next((b for b, v in pub_pcts.items() if round(v, 3) == round(share, 3)), None)
        rows.append({"source": T1, "measure": f"% of calls with duration {band}", "published": own,
                     "rebuilt": share, "difference": share - own,
                     "status": "PUBLISHED LABEL ERROR" if printed_as else "FAIL",
                     "note": f"true value is printed against '{printed_as}'" if printed_as else ""})

### The registered-patient gap

The two patient totals are the only figures that don't rebuild exactly. My 1 August join is
19,523 short on open active practices and 4,333 short on included practices (gate rows
below). Before I accept that, the next cell tests two things.

1. **Am I using the right list?** I hold the 1 July, 1 August and 1 September lists. For the
   July and August call months, I join each list to that month's participation list and
   measure the gap against that month's published totals. If NHS England builds its totals
   on the call month's own list, that list gives the smallest gap. The cell stops if it
   doesn't.
2. **Where is the gap?** I list every participating practice with no row in the call
   month's list: its name, whether it's included, its ODS status, and whether it turns up
   in the list either side.

Patient coverage matches at published precision (0.875) either way, so the gap doesn't move
any published rate. Gold uses the **published** patient totals for national rates and my
1 August join for regional breakdowns, which is the only way to get them by region.

In [5]:
# I hold three registered-patient lists; Bronze writes one folder per list month
REG_LISTS = sorted(p.name for p in (BRONZE_DIR / "registered_patients").iterdir() if p.is_dir())
assert {"2026-07", MONTH, "2026-09"} <= set(REG_LISTS), (
    "I need the 1 July, 1 August and 1 September lists: run "
    "`python src/pipeline/download_bronze.py registered`, then 01_bronze")


def list_sizes(folder: str) -> tuple[pd.Timestamp, pd.Series]:
    t = pd.read_parquet(BRONZE_DIR / "registered_patients" / folder / "gp-reg-pat-prac-all.parquet",
                        columns=["CODE", "EXTRACT_DATE", "NUMBER_OF_PATIENTS"])
    return pd.Timestamp(t.EXTRACT_DATE.iloc[0]), t.set_index("CODE").NUMBER_OF_PATIENTS.astype("int64")


lists = {folder: list_sizes(folder) for folder in REG_LISTS}

# 1. Which list are the published totals built on? I test the July and August call months.
list_test = []
for call_month in ["2026-07", MONTH]:
    p_m = participation[participation.month.eq(call_month)].set_index("practice_code")
    pub_m = (table1[(table1.publication_month == call_month) & (table1.reference_month == call_month)]
             .set_index(["group", "measure"])["value"])
    for folder, (extract_date, sizes) in lists.items():
        matched = p_m.index.to_series().map(sizes)
        list_test.append({
            "call_month": call_month,
            "list_folder": folder,
            "list_date": f"{extract_date:%d %b %Y}",
            "practices_without_list_row": int(matched.isna().sum()),
            "open_active_gap": int(matched.sum() - pub_m[("Coverage", "Registered patients at open active practices")]),
            "included_gap": int(matched[p_m.included].sum() - pub_m[("Coverage", "Registered patients at included practices")]),
        })
list_test = pd.DataFrame(list_test)
list_test["total_abs_gap"] = list_test.open_active_gap.abs() + list_test.included_gap.abs()
closest = list_test.loc[list_test.groupby("call_month").total_abs_gap.idxmin()]
# I stop if any call month is closer to another month's list than to its own
assert (closest.list_folder == closest.call_month).all(), closest
list_test.to_csv(EVIDENCE_DIR / f"gold_patient_list_test_{MONTH}.csv", index=False)
print("Closest list for each call month:", dict(zip(closest.call_month, closest.list_date)))
display(list_test)

# 2. Where is the August gap? I list every participating practice with no 1 August list row.
no_list = practice_dim.loc[~practice_dim.in_registered,
                           ["practice_code", "practice_name", "included", "attribution", "ods_status"]].copy()
for folder in REG_LISTS:
    if folder != MONTH:
        no_list[f"list_size_{folder}"] = no_list.practice_code.map(lists[folder][1]).astype("Int64")
no_list = no_list.sort_values(["included", "practice_name"], ascending=[False, True])
no_list.to_csv(EVIDENCE_DIR / f"gold_practices_without_list_{MONTH}.csv", index=False)
print(f"{len(no_list)} participating practices with no 1 August list row; {int(no_list.included.sum())} included")
no_list

Closest list for each call month: {'2026-07': '01 Jul 2026', '2026-08': '01 Aug 2026'}


,call_month,list_folder,list_date,practices_without_list_row,open_active_gap,included_gap,total_abs_gap
0,2026-07,2026-07,01 Jul 2026,42,-18807,-4365,23172
1,2026-07,2026-08,01 Aug 2026,52,-217407,-199033,416440
2,2026-07,2026-09,01 Sep 2026,54,-335703,-317504,653207
3,2026-08,2026-07,01 Jul 2026,41,152418,158015,310433
4,2026-08,2026-08,01 Aug 2026,42,-19523,-4333,23856
5,2026-08,2026-09,01 Sep 2026,44,-137819,-122055,259874


42 participating practices with no 1 August list row; 1 included


,practice_code,practice_name,included,attribution,ods_status,list_size_2026-07,list_size_2026-09
3318,H84053,BERRYLANDS SURGERY,True,shared_account,DORMANT,<NA>,<NA>
6140,Y06354,ASPIRO HEALTHCARE,False,not_included,ACTIVE,<NA>,<NA>
6107,Y05162,AT MEDICS STREATHAM PCN EA HUB,False,not_included,ACTIVE,<NA>,<NA>
6145,Y06487,BABYLON GP AT HAND BIRMINGHAM,False,not_included,ACTIVE,<NA>,<NA>
6116,Y05258,BARNET FEDERATED GPS LIMITED,False,not_included,ACTIVE,<NA>,<NA>
5947,Y01297,BEACON HOUSE MINISTRIES,False,not_included,ACTIVE,<NA>,<NA>
6135,Y06153,BEDOC HEALTHCARE,False,not_included,ACTIVE,<NA>,<NA>
6115,Y05257,CAMDEN BRIDGING ACCESS SERVICE,False,not_included,ACTIVE,<NA>,<NA>
6105,Y05125,CANNOCK PRACTICES NETWORK SURGERY,False,not_included,ACTIVE,<NA>,<NA>
3895,K84008,CHALGROVE & WATLINGTON SURGERIES,False,not_included,DORMANT,8073,<NA>


**What the test shows.** Each call month's published totals sit closest to that month's own
list: July against 1 July (−18,807 open active, −4,365 included) and August against 1 August
(−19,523 and −4,333). The neighbouring lists are between 137,819 and 335,703 patients out on
the open-active total. NHS England uses the call month's list, and so do I.

The August gap sits in the 42 participating practices with no 1 August list row. 41 aren't
included in the call data. By name, most are services that don't hold a registered list:
extended-access hubs, walk-in and urgent treatment centres, clinics, a COVID medicines
delivery unit, and homeless and vulnerable-adult services. Seven are named like ordinary
surgeries (Chalgrove & Watlington, Manston, Netherton, Penvale Park, Seaforth Village,
Shinfield, Waverley). Two of those seven, Chalgrove & Watlington and Manston, are now dormant,
and only Chalgrove & Watlington has a row in either neighbouring list (8,073 on 1 July).

The one included practice is Berrylands Surgery (H84053), now dormant and on a shared account.
It has no row in any of the three lists. NHS England's included total is 4,333 higher than
mine, so I infer that's the list size it holds for Berrylands. That's an inference: no public
file I hold confirms it. The other 15,190 of the open-active gap sits with the 41 practices
not included, whose list sizes aren't in any public file I hold either.

In [6]:
gate = pd.DataFrame(rows)
explained = gate.measure.str.startswith("Registered patients at") & gate.status.eq("FAIL")
missing = practice_dim.loc[~practice_dim.in_registered]
gate.loc[explained, "status"] = "EXPLAINED"
gate.loc[explained, "note"] = (f"{len(missing)} participating practices have no 1 Aug 2026 list row "
                               f"({int(missing.included.sum())} included); gap is located there")
gate.to_csv(EVIDENCE_DIR / f"gold_reconciliation_{MONTH}.csv", index=False)

print(gate.status.value_counts().to_string())
assert not gate.status.eq("FAIL").any(), gate[gate.status.eq("FAIL")]
print("\nGATE PASSED: every published figure reproduced, or its gap located and explained.")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}" if abs(v) < 10 else f"{v:,.0f}")
gate

status
PASS                     43
PUBLISHED LABEL ERROR     4
EXPLAINED                 2

GATE PASSED: every published figure reproduced, or its gap located and explained.


,source,measure,published,rebuilt,difference,status,note
0,Table 1,Open active practices,"6,171","6,171",0.0000,PASS,
1,Table 1,Count of practices included,"5,327","5,327",0.0000,PASS,
2,Table 1,Practice coverage,0.8630,0.8632,0.0002,PASS,
3,Table 1,Registered patients at open active practices,"63,257,431","63,237,908","-19,523",EXPLAINED,42 participating practices have no 1 Aug 2026 ...
4,Table 1,Registered patients at included practices,"55,333,523","55,329,190","-4,333",EXPLAINED,42 participating practices have no 1 Aug 2026 ...
5,Table 1,Patient coverage,0.8750,0.8749,-0.0001,PASS,
6,Table 1,Number of working weekdays,20,20,0.0000,PASS,
7,Table 1,Total inbound calls,"27,160,966","27,160,966",0.0000,PASS,
8,Table 1,"Rate of inbound calls per 1,000 registered pat...",491,491,-0.0024,PASS,
9,Table 1,Calls answered,"15,675,776","15,675,776",0.0000,PASS,


## Step 3 — What the gate found about the Monday figure

The gate passes, but it shows the headlines don't all treat **Monday 31 August**, the
Summer Bank Holiday, the same way:

| Headline | Includes the bank holiday? |
|---|---|
| "2,037,943 (7.5%) of inbound calls were made between 8am and 10am on Monday mornings" | **Yes**: 70,616 of those calls were on 31 August |
| 25,039,293 inbound in core hours (Table 4b) | **No** |
| 6,801,853 inbound 8am–10am Mon–Fri (Table 4b) | **No** |
| Working weekdays = 20 (Table 1) | **No** |

So the Monday figure adds up four ordinary Mondays and a bank holiday, and August had five
Mondays, one more than most months. Below I separate the two and put the Monday peak on a
per-day basis that compares fairly across months.

In [7]:
mon = inbound[inbound.weekday.eq("Monday")]
per_monday = (mon[mon.time_band.eq("08:00-09:59")].groupby(["date", "is_bank_holiday"])["calls"].sum()
              .reset_index().rename(columns={"calls": "calls_08_10"}))
per_monday["calls_whole_day"] = per_monday.date.map(mon.groupby("date")["calls"].sum())
normal = per_monday[~per_monday.is_bank_holiday]

weekday_avg_0810 = (weekday_core[weekday_core.time_band.eq("08:00-09:59")]
                    .groupby(["weekday", "date"], observed=True)["calls"].sum()
                    .groupby("weekday", observed=True).mean().reindex(WEEKDAYS[:5]))
monday_summary = pd.DataFrame({
    "measure": [
        "Monday 08:00-10:00, published (5 Mondays incl. bank holiday)",
        "...of which the bank holiday (31 Aug)",
        "Monday 08:00-10:00, 4 ordinary Mondays",
        "Average ordinary Monday, 08:00-10:00",
        "Average ordinary Tuesday-Friday, 08:00-10:00",
        "Ordinary Monday vs Tue-Fri average, 08:00-10:00",
        "Share of an ordinary Monday's calls arriving 08:00-10:00",
    ],
    "value": [
        mon_8_10,
        int(per_monday.loc[per_monday.is_bank_holiday, "calls_08_10"].sum()),
        int(normal.calls_08_10.sum()),
        normal.calls_08_10.mean(),
        weekday_avg_0810.iloc[1:].mean(),
        normal.calls_08_10.mean() / weekday_avg_0810.iloc[1:].mean(),
        normal.calls_08_10.sum() / normal.calls_whole_day.sum(),
    ],
})
monday_summary.to_csv(GOLD_DIR / f"monday_0810_{MONTH}.csv", index=False)
display_monday = monday_summary.copy()
display_monday["value"] = [f"{v:,.0f}" if v > 10 else (f"{v:.1%}" if v < 1 else f"{v:.2f}x") for v in monday_summary.value]
display_monday

,measure,value
0,"Monday 08:00-10:00, published (5 Mondays incl....","2,037,943"
1,...of which the bank holiday (31 Aug),"70,616"
2,"Monday 08:00-10:00, 4 ordinary Mondays","1,967,327"
3,"Average ordinary Monday, 08:00-10:00","491,832"
4,"Average ordinary Tuesday-Friday, 08:00-10:00","302,158"
5,"Ordinary Monday vs Tue-Fri average, 08:00-10:00",1.63x
6,Share of an ordinary Monday's calls arriving 0...,28.7%


### The week in 2-hour bands (August 2026)

Average inbound calls per day for each weekday × time band, with the bank holiday kept
out of the Monday average and shown as its own row. Weekend bands are tiny, so the grid
is mostly a picture of the working week. This is the table behind the 7 × 9 chart.

In [8]:
inbound["day_label"] = np.where(inbound.is_bank_holiday, "Monday (bank holiday)", inbound.weekday.astype(str))
days_per_label = inbound.groupby("day_label")["date"].nunique()
grid = (inbound.groupby(["day_label", "time_band"], observed=True)["calls"].sum()
        .unstack("time_band"))
grid = grid.div(days_per_label, axis=0).round(0).astype(int)
grid = grid.reindex(WEEKDAYS + ["Monday (bank holiday)"])
grid.insert(0, "days", days_per_label.reindex(grid.index))
grid.to_csv(GOLD_DIR / f"day_band_grid_{MONTH}.csv")
grid

time_band,days,00:00-05:59,06:00-07:59,08:00-09:59,10:00-11:59,12:00-13:59,14:00-15:59,16:00-17:59,18:00-18:29,18:30-23:59
day_label,,,,,,,,,,
Monday,4,1142,95166,491832,374319,272206,279868,176671,12797,11812
Tuesday,4,1034,59118,334128,297150,227208,237186,153818,11261,10876
Wednesday,4,996,53134,302889,276007,212809,225259,144458,10411,10333
Thursday,4,1020,49432,284279,265204,204041,213922,137776,9908,10275
Friday,4,1052,51256,287336,261378,201236,214300,130692,9475,9499
Saturday,5,748,2159,19164,18803,10568,7103,4271,657,3287
Sunday,5,610,925,4013,4338,3513,3210,2607,556,3872
Monday (bank holiday),1,681,9335,70616,47146,31565,24879,14646,1450,4757


## Step 4 — Coverage by patients: who sits behind the missing 13.7%?

The 844 open, active practices not included in August 2026 fall into two groups that mean
different things:

- **Didn't agree to take part** (64 practices): outside the collection by choice.
- **Agreed, but no data reaching the publication** (780 practices): mostly a supplier
  that can't submit yet. The Supporting Information page names suppliers still developing
  their submissions.

I weight each by registered patients (1 August list) and break it down by NHS England
region. Practices with no list row count as practices but add no patients, so they can't
inflate the patient share.

In [9]:
pdim = practice_dim.copy()
pdim["status"] = np.select([pdim.included, pdim.agreed], ["included", "agreed_not_included"], "not_agreed")
pdim["patients"] = pdim.registered_patients.fillna(0)

def coverage_table(frame, by):
    t = frame.pivot_table(index=by, columns="status", values="patients", aggfunc="sum", fill_value=0)
    n = frame.pivot_table(index=by, columns="status", values="practice_code", aggfunc="count", fill_value=0)
    out = pd.DataFrame({
        "practices": n.sum(axis=1),
        "practices_included": n.get("included", 0),
        "practice_coverage": n.get("included", 0) / n.sum(axis=1),
        "patients": t.sum(axis=1),
        "patients_not_agreed": t.get("not_agreed", 0),
        "patients_agreed_not_included": t.get("agreed_not_included", 0),
        "patient_coverage": t.get("included", 0) / t.sum(axis=1),
    })
    return out

by_region = coverage_table(pdim, "region_name")
national_row = coverage_table(pdim.assign(region_name="ENGLAND"), "region_name")
coverage = pd.concat([by_region.sort_values("patient_coverage"), national_row])
coverage["patients_outside"] = coverage.patients_not_agreed + coverage.patients_agreed_not_included
coverage.to_csv(GOLD_DIR / f"coverage_by_region_{MONTH}.csv")
coverage.style.format({"practice_coverage": "{:.1%}", "patient_coverage": "{:.1%}", "patients": "{:,.0f}",
                       "patients_not_agreed": "{:,.0f}", "patients_agreed_not_included": "{:,.0f}",
                       "patients_outside": "{:,.0f}"})

,practices,practices_included,practice_coverage,patients,patients_not_agreed,patients_agreed_not_included,patient_coverage,patients_outside
region_name,,,,,,,,
NORTH EAST AND YORKSHIRE,922,747,81.0%,"9,213,228",91,"1,719,587",81.3%,"1,719,678"
EAST OF ENGLAND,637,514,80.7%,"7,312,182",498,"1,227,064",83.2%,"1,227,562"
NORTH WEST,939,786,83.7%,"7,943,252",121,"1,074,827",86.5%,"1,074,948"
SOUTH EAST,774,666,86.0%,"9,800,294","95,659","1,154,201",87.2%,"1,249,860"
MIDLANDS,1241,1100,88.6%,"11,931,108",34,"1,375,595",88.5%,"1,375,629"
SOUTH WEST,513,457,89.1%,"6,114,308",174,"619,427",89.9%,"619,601"
LONDON,1145,1057,92.3%,"10,923,536","8,968","632,472",94.1%,"641,440"
ENGLAND,6171,5327,86.3%,"63,237,908","105,545","7,803,173",87.5%,"7,908,718"


### Coverage by deprivation

Region is one lens. The other one the README asks about is deprivation: are the practices
outside the data more likely to serve deprived populations? If they are, the national
rates would be tilted towards less deprived patients.

**Quintiles.** I rank all 6,143 practices with a Fingertips IMD 2025 score and split them
into five equal groups of practices. **Q1 is the most deprived fifth, Q5 the least.** The
cut points come from the national practice distribution, not just this month's list, so
they don't move when a practice joins or leaves. The 77 participating practices with no
score (12,197 patients) form their own row.

In [10]:
imd = pd.read_parquet(SILVER_DIR / "practice_imd2025.parquet")
QUINTILES = ["Q1 most deprived", "Q2", "Q3", "Q4", "Q5 least deprived"]
cuts = imd.imd2025_score.quantile([0.2, 0.4, 0.6, 0.8]).tolist()
def quintile(score):
    if pd.isna(score):
        return "No score"
    return QUINTILES[4 - sum(score > c for c in cuts)]  # the more cut points a score exceeds, the more deprived I rank the practice
pdim["imd_quintile"] = pdim.imd2025_score.map(quintile)
assert imd.imd2025_score.map(quintile).value_counts().between(len(imd) // 5 - 5, len(imd) // 5 + 5).all()

by_imd = coverage_table(pdim, "imd_quintile").reindex(QUINTILES + ["No score"])
by_imd["patients_outside"] = by_imd.patients_not_agreed + by_imd.patients_agreed_not_included
ascending = [f"{lo:.1f} to {hi:.1f}" for lo, hi in zip([imd.imd2025_score.min()] + cuts, cuts + [imd.imd2025_score.max()])]
by_imd["score_range"] = ascending[::-1] + [""]  # I reverse the ranges because Q1 holds the highest (most deprived) scores
assert pdim.loc[pdim.imd_quintile.eq("Q1 most deprived"), "imd2025_score"].min() >= cuts[-1]
by_imd.to_csv(GOLD_DIR / f"coverage_by_deprivation_{MONTH}.csv")
by_imd.style.format({"practice_coverage": "{:.1%}", "patient_coverage": "{:.1%}", "patients": "{:,.0f}",
                     "patients_not_agreed": "{:,.0f}", "patients_agreed_not_included": "{:,.0f}",
                     "patients_outside": "{:,.0f}"})

,practices,practices_included,practice_coverage,patients,patients_not_agreed,patients_agreed_not_included,patient_coverage,patients_outside,score_range
imd_quintile,,,,,,,,,
Q1 most deprived,1218,1016,83.4%,"10,202,961",0,"1,696,903",83.4%,"1,696,903",33.3 to 65.4
Q2,1217,1065,87.5%,"12,495,628","18,356","1,598,532",87.1%,"1,616,888",24.8 to 33.3
Q3,1217,1059,87.0%,"13,220,103","2,593","1,778,392",86.5%,"1,780,985",18.4 to 24.8
Q4,1219,1091,89.5%,"13,107,699","6,913","1,214,411",90.7%,"1,221,324",13.0 to 18.4
Q5 least deprived,1223,1086,88.8%,"14,199,320","75,313","1,510,741",88.8%,"1,586,054",3.9 to 13.0
No score,77,10,13.0%,"12,197","2,370","4,194",46.2%,"6,564",


### Is that just region in disguise?

The most deprived fifth has the lowest patient coverage, but deprivation and region are
tangled. North East and Yorkshire has the lowest coverage of any region and a large
deprived population. So I compare Q1 with Q2–Q5 **within each region**, then ask: if Q1
patients had the coverage of the less deprived practices in their own region, what would
Q1 coverage be?

In [11]:
scored = pdim[pdim.imd2025_score.notna()].assign(q1=lambda t: t.imd_quintile.eq("Q1 most deprived"))
def cov(g):
    return (g.patients * g.included).sum() / g.patients.sum()
within = scored.groupby(["region_name", "q1"]).apply(cov, include_groups=False).unstack()
within.columns = ["coverage_Q2_to_Q5", "coverage_Q1"]
within["gap_points"] = (within.coverage_Q1 - within.coverage_Q2_to_Q5) * 100
within["Q1_share_of_region_patients"] = scored.groupby("region_name").apply(
    lambda g: g.loc[g.q1, "patients"].sum() / g.patients.sum(), include_groups=False)
q1 = scored[scored.q1]
q1_weights = q1.groupby("region_name").patients.sum()
expected = (q1_weights * within.coverage_Q2_to_Q5).sum() / q1_weights.sum()
actual = cov(q1)
# I split the raw Q1 gap into the part its regional mix accounts for and the part left over
rest = cov(scored[~scored.q1])
# I give each region its share of the leftover: its share of Q1 patients x its own Q1 gap. These sum to the leftover.
within["Q1_share_of_all_Q1_patients"] = q1_weights / q1_weights.sum()
within["contribution_to_gap_points"] = within.Q1_share_of_all_Q1_patients * within.gap_points
assert abs(within.contribution_to_gap_points.sum() - (actual - expected) * 100) < 1e-9
print(f"Q1 patient coverage {actual:.1%} vs Q2-Q5 {rest:.1%}: raw gap {(actual - rest) * 100:+.1f} points")
print(f"  explained by Q1's regional mix: {(expected - rest) * 100:+.1f} points (Q1 would be {expected:.1%})")
print(f"  left over within regions:       {(actual - expected) * 100:+.1f} points")
within.to_csv(GOLD_DIR / f"deprivation_gap_within_region_{MONTH}.csv")
within.sort_values("contribution_to_gap_points").style.format({
    "coverage_Q2_to_Q5": "{:.1%}", "coverage_Q1": "{:.1%}", "gap_points": "{:+.1f}",
    "Q1_share_of_region_patients": "{:.1%}", "Q1_share_of_all_Q1_patients": "{:.1%}",
    "contribution_to_gap_points": "{:+.2f}"})

Q1 patient coverage 83.4% vs Q2-Q5 88.3%: raw gap -4.9 points
  explained by Q1's regional mix: -1.4 points (Q1 would be 86.9%)
  left over within regions:       -3.5 points


,coverage_Q2_to_Q5,coverage_Q1,gap_points,Q1_share_of_region_patients,Q1_share_of_all_Q1_patients,contribution_to_gap_points
region_name,,,,,,
NORTH WEST,89.3%,80.9%,-8.4,33.5%,26.1%,-2.20
MIDLANDS,90.1%,82.7%,-7.4,21.6%,25.3%,-1.87
SOUTH EAST,87.9%,66.9%,-21.1,3.2%,3.1%,-0.65
LONDON,94.3%,92.3%,-2.0,9.0%,9.7%,-0.19
EAST OF ENGLAND,83.5%,80.2%,-3.2,6.9%,5.0%,-0.16
SOUTH WEST,89.7%,96.2%,+6.5,3.0%,1.8%,+0.12
NORTH EAST AND YORKSHIRE,79.7%,84.7%,+5.0,32.2%,29.1%,+1.44


**Reading it:** region explains part of the gap, not all of it. Q1 patient coverage is 83.4%
against 88.3% for Q2–Q5, a gap of 4.9 points. If Q1 patients had the Q2–Q5 coverage of their
own region, Q1 would be at 86.9%. So Q1's regional mix accounts for 1.4 points, and 3.5 points
remain within regions.

Two regions carry the remainder. In the North West and the Midlands, Q1 practices are 8.4 and
7.4 points less covered than the other practices in their region. Between them they hold just
over half of all Q1 patients, and together they contribute −4.1 points, more than the whole
remainder. North East and Yorkshire runs the other way (+5.0 points within the region), which
offsets 1.4 points. The South East gap is large (−21.1 points), but its Q1 practices hold only
3% of Q1 patients, so it adds −0.6.

This is about the **data**, not about access. These are practices whose phone supplier
isn't yet sending data to NHS England, not practices that answer fewer calls. But it
means the national headline rates under-represent the most deprived fifth of practices,
and that's worth saying whenever they're quoted.

### Call outcomes by deprivation

Same quintiles, now for what happens to calls. This uses only practice-attributed calls,
because shared-account calls can't be given a practice score. Calls are summed within
each quintile before dividing, so large practices count in proportion to their calls.
This compares groups of practices, not individual practices.

In [12]:
oc_all = pd.read_parquet(SILVER_DIR / "practice_outcome_check.parquet")
oc_all = oc_all[oc_all.month.eq(MONTH)].merge(pdim[["practice_code", "imd_quintile", "patients"]], on="practice_code", how="left")
q = oc_all.groupby("imd_quintile")[["CBT001", "CBT002", "CBT003", "CBT004", "CBT005", "CBT007", "patients"]].sum().reindex(QUINTILES + ["No score"])
outcomes_imd = pd.DataFrame({
    "practices": oc_all.groupby("imd_quintile").size().reindex(q.index),
    "inbound": q.CBT001,
    "calls_per_1000_patients_per_working_day": q.CBT001 / q.patients * 1000 / working_weekdays,
    "answered": q.CBT007 / q.CBT001,
    "ended_in_ivr": q.CBT002 / q.CBT001,
    "callback_requested": q.CBT005 / q.CBT001,
    "missed": q.CBT004 / q.CBT001,
})
# I blank the rate for no-score practices: they hold almost no registered list, so a per-patient rate would mean nothing
outcomes_imd.loc["No score", "calls_per_1000_patients_per_working_day"] = np.nan
outcomes_imd.to_csv(GOLD_DIR / f"outcomes_by_deprivation_{MONTH}.csv")
outcomes_imd.style.format({"inbound": "{:,.0f}", "calls_per_1000_patients_per_working_day": "{:.1f}",
                           "answered": "{:.1%}", "ended_in_ivr": "{:.1%}", "callback_requested": "{:.1%}", "missed": "{:.1%}"})

,practices,inbound,calls_per_1000_patients_per_working_day,answered,ended_in_ivr,callback_requested,missed
imd_quintile,,,,,,,
Q1 most deprived,914,"4,434,008",28.7,59.0%,24.3%,6.0%,10.8%
Q2,982,"4,926,581",25.0,57.6%,25.6%,6.6%,10.2%
Q3,1005,"5,321,184",24.7,56.8%,25.7%,7.1%,10.4%
Q4,1029,"5,310,722",23.5,57.7%,25.1%,6.7%,10.5%
Q5 least deprived,1053,"5,404,052",22.1,58.9%,24.5%,6.5%,10.0%
No score,6,"15,241",nan,78.5%,14.5%,0.3%,6.6%


## Step 5 — The 31.9%: calls that are neither answered nor missed

Nationally, 57.7% of inbound calls are answered and 10.4% are missed. The other 31.9% are
calls that ended in the automated IVR stage (25.3%) or became a callback request (6.6%).
Both count as "dealt with" in NHS England's definition, but neither is a conversation with
practice staff at the time of the call.

By region, I use every call that can be placed there. That's practice-attributed calls
plus shared-account calls whose region is filled in (Silver Step 2). The 9.4% of
shared-account calls with no region form an "Unmapped" row, so the regions still add up
to England.

In [13]:
OUTCOMES = {"CBT003": "answered", "CBT002": "ended_in_ivr", "CBT005": "callback_requested", "CBT004": "missed"}
dd = durations[durations.indicator.isin(["CBT001", *OUTCOMES])]
reg_names = practice_dim.drop_duplicates("region_code").set_index("region_code")["region_name"]
dd = dd.assign(region=dd.region_code.astype(str).map(reg_names).fillna("UNMAPPED (shared accounts)"))
w = dd.groupby(["region", "indicator"], observed=True)["calls"].sum().unstack()
w.loc["ENGLAND"] = w.sum()
outcomes = pd.DataFrame({"inbound": w["CBT001"]})
for c, name in OUTCOMES.items():
    outcomes[name] = w[c] / w["CBT001"]
outcomes["neither_answered_nor_missed"] = outcomes.ended_in_ivr + outcomes.callback_requested
outcomes["outcome_sum_vs_inbound"] = w[list(OUTCOMES)].sum(axis=1) / w["CBT001"] - 1
outcomes = pd.concat([outcomes.drop("ENGLAND").sort_values("ended_in_ivr"), outcomes.loc[["ENGLAND"]]])
outcomes.to_csv(GOLD_DIR / f"outcomes_by_region_{MONTH}.csv")
outcomes.style.format({c: "{:.1%}" for c in outcomes.columns if c != "inbound"} | {"inbound": "{:,.0f}",
                       "outcome_sum_vs_inbound": "{:+.3%}"})

,inbound,answered,ended_in_ivr,callback_requested,missed,neither_answered_nor_missed,outcome_sum_vs_inbound
region,,,,,,,
NORTH EAST AND YORKSHIRE,"4,518,109",57.2%,23.7%,6.8%,12.2%,30.5%,+0.008%
MIDLANDS,"5,533,100",56.9%,24.1%,7.4%,11.6%,31.5%,+0.001%
SOUTH WEST,"2,601,907",57.6%,24.2%,7.7%,10.6%,31.8%,+0.001%
NORTH WEST,"3,764,751",61.2%,24.5%,5.8%,8.5%,30.3%,+0.007%
UNMAPPED (shared accounts),"163,693",56.2%,25.7%,6.7%,11.4%,32.4%,+0.000%
LONDON,"4,160,357",61.3%,26.3%,4.5%,7.9%,30.8%,+0.001%
EAST OF ENGLAND,"2,822,650",54.9%,27.0%,6.2%,11.9%,33.2%,+0.002%
SOUTH EAST,"3,596,399",54.1%,28.2%,8.1%,9.6%,36.3%,+0.003%
ENGLAND,"27,160,966",57.7%,25.3%,6.6%,10.4%,31.9%,+0.003%


### How much does the IVR share vary between practices?

A regional average can hide a wide spread. Practices configure their own phone systems
(the Supporting Information page warns against comparing them for this reason), so the
IVR share is partly a design choice: some practices use it to signpost, some barely use
it. Practices with fewer than 500 inbound calls in the month are left out, so small
denominators don't stretch the spread.

In [14]:
oc = pd.read_parquet(SILVER_DIR / "practice_outcome_check.parquet")
oc = oc[oc.month.eq(MONTH) & (oc.CBT001 >= 500)].copy()
oc["ivr_share"] = oc.CBT002 / oc.CBT001
oc["answered_share"] = oc.CBT003 / oc.CBT001
oc["missed_share"] = oc.CBT004 / oc.CBT001
spread = oc[["ivr_share", "answered_share", "missed_share"]].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T
spread["practices_with_zero"] = [(oc[c] == 0).sum() for c in spread.index]
spread.to_csv(GOLD_DIR / f"practice_outcome_spread_{MONTH}.csv")
spread[["count", "10%", "25%", "50%", "75%", "90%", "practices_with_zero"]].style.format(
    {c: "{:.1%}" for c in ["10%", "25%", "50%", "75%", "90%"]} | {"count": "{:,.0f}"})

,count,10%,25%,50%,75%,90%,practices_with_zero
ivr_share,"4,985",13.6%,17.7%,22.7%,28.3%,34.3%,45
answered_share,"4,985",43.1%,54.3%,63.8%,71.7%,77.5%,2
missed_share,"4,985",3.2%,5.1%,8.0%,12.4%,18.0%,0


### The 31.9% through the day

A national split hides when things happen. Here I take all four outcomes for every weekday and
two-hour band, using every call (shared-account calls included, as in NHS England's national
tables). The bank holiday stays out of Monday and gets its own row, and Tuesday to Friday also
appear as one group so Monday has a single comparison. Calls sit in the band of the time they
were made (Supporting Information).

Before reading anything off it, I tie the table back to the gate: inbound must equal the
published total, Monday 08:00–10:00 must equal the published 2,037,943, and the outcomes must
over-count by the same 908 calls as they do nationally.

In [15]:
# I use every call here, practice and shared-account alike, as NHS England's national tables do
OUTCOME_CODES = {"CBT003": "answered", "CBT002": "ended_in_ivr", "CBT005": "callback_requested", "CBT004": "missed"}
dt_out = day_time[day_time.indicator.isin(["CBT001", *OUTCOME_CODES])].copy()
dt_out["day_label"] = np.where(dt_out.is_bank_holiday, "Monday (bank holiday)", dt_out.weekday.astype(str))
# I add Tuesday to Friday as one group, so Monday has a single comparison row
tue_fri = dt_out[dt_out.day_label.isin(["Tuesday", "Wednesday", "Thursday", "Friday"])].assign(day_label="Tuesday-Friday")
counts = (pd.concat([dt_out, tue_fri])
          .pivot_table(index=["day_label", "time_band"], columns="indicator", values="calls", aggfunc="sum", observed=True))

band_outcomes = pd.DataFrame({"inbound": counts["CBT001"]})
for code_, name in OUTCOME_CODES.items():
    band_outcomes[name] = counts[code_] / counts["CBT001"]
band_outcomes["outcome_sum_vs_inbound"] = counts[list(OUTCOME_CODES)].sum(axis=1) / counts["CBT001"] - 1

# I tie the table back to the gate before reading anything off it
days = band_outcomes.drop(index="Tuesday-Friday", level="day_label")
assert int(days.inbound.sum()) == total, "inbound doesn't match the gate total"
assert int(counts.loc[("Monday", "08:00-09:59"), "CBT001"]
           + counts.loc[("Monday (bank holiday)", "08:00-09:59"), "CBT001"]) == mon_8_10, "Monday 8-10 doesn't match the gate"
over_count = int(counts.drop(index="Tuesday-Friday", level="day_label")[list(OUTCOME_CODES)].sum().sum() - total)
print(f"Outcomes over-count inbound by {over_count:,} calls in total (national figure: "
      f"{int(nat[list(OUTCOME_CODES)].sum() - nat['CBT001']):,}); "
      f"largest cell gap {days.outcome_sum_vs_inbound.abs().max():.2%}")
band_outcomes.to_csv(GOLD_DIR / f"outcomes_by_day_band_{MONTH}.csv")

# I show the core-hours bands, ordinary Monday against Tuesday-Friday
view = band_outcomes.loc[(["Monday", "Tuesday-Friday"], CORE_BANDS), ["inbound", *OUTCOME_CODES.values()]]
view.style.format({"inbound": "{:,.0f}"} | {c: "{:.1%}" for c in OUTCOME_CODES.values()})

Outcomes over-count inbound by 908 calls in total (national figure: 908); largest cell gap 0.05%


**Reading it:** on its own, the missed rate is misleading here, because it's lowest exactly when
the line is busiest. On an ordinary Monday, 11.1% of 08:00–10:00 calls are missed, against
12.8% from 10:00 to 12:00. But only 48.7% are answered at the time, the only core-hours band
below half, because 14.5% become callback requests, against 8.8% or less later on a Monday.
The queue doesn't shrink at 8am. Part of it moves into callbacks.

Tuesday to Friday has the same shape, less sharply: 59.8% answered and 9.7% callback requests
at 08:00–10:00, against 65.8% and 5.8% from 10:00 to 12:00.

Two caveats travel with this. Missed includes voicemail, and practices set up queues and
callbacks differently, so this describes the national mix of phone systems, not how any one
practice handles its 8am rush. And a callback request isn't a failure: 99.6% of requested
callbacks are made (gate).

## Step 6 — Eleven months on a like-for-like basis

Monthly call totals rise mostly because more practices join (the Supporting Information
page says so), so raw totals aren't a demand trend. For each month I put inbound calls
per 1,000 registered patients at included practices, and per working weekday, side by
side. Patient denominators come from the **August 2026 edition** of Table 1, because
earlier editions were revised (Silver Step 8).

Wait times are flagged from June 2026, when a supplier corrected its wait-time method and
earlier months weren't restated (Table 1 note 5). The "% answered under 2 minutes" series
isn't comparable across that line.

In [16]:
aug_ed = table1[table1.publication_month.eq(MONTH)].pivot_table(index="reference_month", columns=["group", "measure"], values="value")
assert not table1[table1.publication_month.eq(MONTH)].duplicated(["reference_month", "group", "measure"]).any()
UNDER_2 = next(c for c in aug_ed.columns if c[0] == "Call wait times (any time)" and c[1].startswith("Percentage of calls answered after waiting under 2"))
months = sorted(participation.month.unique())
trend = []
# I read the national totals once; the loop only filters them by month
nat_by_file = pd.read_csv(SILVER_DIR / "national_by_indicator_and_file.csv")
for m in months:
    nat_m = nat_by_file[nat_by_file.month.eq(m)].set_index("indicator")["durations"]
    pts = aug_ed.loc[m, ("Coverage", "Registered patients at included practices")]
    wd = aug_ed.loc[m, ("Working Days", "Number of working weekdays")]
    trend.append({
        "month": m,
        "practices_included": int(aug_ed.loc[m, ("Coverage", "Count of practices included")]),
        "patient_coverage": aug_ed.loc[m, ("Coverage", "Patient coverage")],
        "inbound_calls": int(nat_m["CBT001"]),
        "calls_per_1000_patients": nat_m["CBT001"] / pts * 1000,
        "working_weekdays": int(wd),
        "calls_per_1000_patients_per_working_day": nat_m["CBT001"] / pts * 1000 / wd,
        "answered": nat_m["CBT007"] / nat_m["CBT001"],
        "ended_in_ivr": nat_m["CBT002"] / nat_m["CBT001"],
        "callback_requested": nat_m["CBT005"] / nat_m["CBT001"],
        "missed": nat_m["CBT004"] / nat_m["CBT001"],
        "answered_under_2min_published": aug_ed.loc[m, UNDER_2],
        "wait_time_series_break": m >= "2026-06",
    })
trend = pd.DataFrame(trend).set_index("month")

# I check whether each month's CSVs still agree with the latest (August 2026) edition of Table 1
latest = aug_ed.loc[months, [("Inbound calls", "Total Inbound Calls"), ("Calls dealt with", "Calls answered [2]")]]
latest.columns = ["inbound_latest_edition", "answered_latest_edition"]
nat_all = nat_by_file.pivot(index="month", columns="indicator", values="durations")
vs_latest = latest.join(nat_all[["CBT001", "CBT007"]].rename(columns={"CBT001": "inbound_csv", "CBT007": "answered_csv"}))
vs_latest["inbound_revision"] = vs_latest.inbound_latest_edition - vs_latest.inbound_csv
vs_latest["answered_revision"] = vs_latest.answered_latest_edition - vs_latest.answered_csv
revised_months = vs_latest[(vs_latest.inbound_revision != 0) | (vs_latest.answered_revision != 0)]
KNOWN_REVISIONS = ["2025-11"]  # I allow only Nov 2025: restated upward from the Jan 2026 edition, and its CSVs were never reissued
assert set(revised_months.index) <= set(KNOWN_REVISIONS), revised_months
trend["revised_after_publication"] = trend.index.isin(revised_months.index)
print("Months whose call totals were revised after their CSVs were published:")
print(revised_months[["inbound_revision", "answered_revision"]].to_string())
trend.to_csv(GOLD_DIR / "monthly_trend.csv")
trend.style.format({"patient_coverage": "{:.1%}", "inbound_calls": "{:,.0f}", "calls_per_1000_patients": "{:.1f}",
                    "calls_per_1000_patients_per_working_day": "{:.2f}", "answered": "{:.1%}", "ended_in_ivr": "{:.1%}",
                    "callback_requested": "{:.1%}", "missed": "{:.1%}", "answered_under_2min_published": "{:.1%}"})

Months whose call totals were revised after their CSVs were published:
                 inbound_revision  answered_revision
reference_month                                     
2025-11                    29,234             15,431


,practices_included,patient_coverage,inbound_calls,calls_per_1000_patients,working_weekdays,calls_per_1000_patients_per_working_day,answered,ended_in_ivr,callback_requested,missed,answered_under_2min_published,wait_time_series_break,revised_after_publication
month,,,,,,,,,,,,,
2025-10,4649,75.7%,"31,444,618",649.9,23,28.26,54.8%,26.0%,7.3%,11.8%,57.9%,False,False
2025-11,5019,82.3%,"29,439,840",559.4,20,27.97,54.1%,26.4%,7.6%,11.9%,58.3%,False,True
2025-12,5093,83.5%,"30,063,218",562.9,21,26.81,54.8%,26.8%,7.3%,11.0%,59.5%,False,False
2026-01,5109,83.9%,"31,605,636",589.6,21,28.08,55.0%,26.5%,7.6%,10.9%,59.2%,False,False
2026-02,5112,83.9%,"29,211,036",545.7,20,27.28,55.2%,26.5%,7.5%,10.8%,59.6%,False,False
2026-03,5245,85.8%,"33,483,741",612.4,22,27.84,54.2%,27.3%,7.4%,11.1%,59.6%,False,False
2026-04,5292,86.4%,"30,901,353",561.7,20,28.09,54.3%,26.8%,7.4%,11.5%,59.2%,False,False
2026-05,5290,86.5%,"28,491,203",517.7,19,27.25,54.8%,26.6%,7.3%,11.3%,59.3%,False,False
2026-06,5292,86.7%,"33,128,213",601.8,22,27.35,56.3%,25.0%,6.9%,11.7%,60.4%,True,False


**Revisions:** every month's call totals, rebuilt from that month's own CSVs, equal the
latest edition of Table 1, with one exception. **November 2025** was restated upward
from the January 2026 edition: +29,234 inbound and +15,431 answered calls. The November
CSVs were never reissued, so the practice-level files and the headline table now disagree
for that month. The trend uses the CSVs (the only practice-level source) and flags the
month in `revised_after_publication`. The difference is 0.1% of the month's calls.

## Step 7 — Call durations, with the labels put right

The true share of answered calls in each duration band, rebuilt from CBT007 for every
month. In the editions from January 2026 onward, these values are printed one row off.

In [17]:
dur_rows = []
for m in months:
    d = pd.read_parquet(SILVER_DIR / "calls_durations" / f"{m}.parquet")
    d = d[d.indicator.eq("CBT007")]
    counts = {k: d.loc[(d.lower_s >= lo) & ((d.upper_s <= hi) if hi else d.upper_s.isna()), "calls"].sum()
              for k, (lo, hi) in DUR_BANDS.items()}
    tot = sum(counts.values())
    dur_rows.append({"month": m, **{k: v / tot for k, v in counts.items()}, "answered_calls": tot})
duration_shares = pd.DataFrame(dur_rows).set_index("month")
duration_shares.to_csv(GOLD_DIR / "call_duration_shares.csv")
duration_shares.style.format({k: "{:.1%}" for k in DUR_BANDS} | {"answered_calls": "{:,.0f}"})

,1 minute or less,1 to 2 minutes,2 to 5 minutes,over 5 minutes,answered_calls
month,,,,,
2025-10,23.5%,36.0%,32.9%,7.6%,"17,239,997"
2025-11,23.4%,35.9%,33.0%,7.8%,"15,936,380"
2025-12,23.9%,36.0%,32.5%,7.6%,"16,479,676"
2026-01,23.1%,36.1%,33.0%,7.8%,"17,391,539"
2026-02,23.1%,35.9%,33.2%,7.9%,"16,124,647"
2026-03,23.6%,35.7%,32.8%,7.8%,"18,150,856"
2026-04,23.4%,35.6%,33.0%,8.0%,"16,780,990"
2026-05,23.2%,35.6%,33.2%,8.0%,"15,599,188"
2026-06,23.8%,35.5%,32.8%,7.9%,"18,663,722"


## What Gold says, in plain terms

**The gate.** Of the 49 published August 2026 figures I can rebuild, 43 match, 4 are published
label errors (the duration percentages, printed one row out of place from the January 2026
edition), and 2 are explained (the registered-patient totals, short by 0.03%, with the gap
located in the 42 practices with no 1 August list row). No figure below leaves Gold without
that gate passing.

**1. Who sits behind the missing 13.7% of practices?** By patients, coverage is 87.5%.
7.91m patients are outside the data on my 1 August join, and 7.80m of them are at practices
that agreed to take part but whose data isn't published. Coverage runs from 81.3% (North East
and Yorkshire) to 94.1% (London). The most deprived fifth of practices is the least covered:
83.4% against 88.3% for the other four fifths. Region accounts for 1.4 points of that gap and
3.5 remain, mostly in the North West and the Midlands.

**2. What happens to the 31.9% that are neither answered nor missed?** 25.3% of calls end in
the IVR before joining a queue and 6.6% become callback requests. Through the day, the mix
shifts: on an ordinary Monday, 08:00–10:00 is the only core-hours band where under half of
calls are answered at the time (48.7%), because 14.5% become callback requests.

**3. What does the Monday 08:00–10:00 figure measure?** The published 2,037,943 rebuilds
exactly, but it adds four ordinary Mondays to the 31 August bank holiday (70,616). An ordinary
Monday has 491,832 calls in those two hours, 1.63 times the Tuesday-to-Friday average and
28.7% of that day's calls.

**Also found on the way:** November 2025's call totals were revised upward after publication
(+29,234 inbound) and its CSVs weren't reissued; and wait-time figures break in June 2026.

**Where the figures live.** Every number above comes from a file in `data/gold/` or
`evidence/` written by this notebook. The README and `docs/sources.md` carry the same figures
with their caveats.

**Still open:**
- The September 2026 release, which will show whether the duration labels are still wrong,
  how the next bank holiday is treated, and whether the summer dip in calls per patient
  was seasonal.
- How Fingertips calculates indicator 94240 (practice IMD 2025). I assume it's weighted by
  where registered patients live, which is NHS England's documented method, but it isn't
  confirmed.
- Whether coverage skews by telephony supplier. There's no public register of which
  supplier each practice uses, so it can't be tested from public data.
- The list sizes NHS England uses for the 42 practices with no 1 August list row aren't in
  any public file I hold.